# Problem Set 1 (Conceptual problems)
**Learning from data [TIF285], Chalmers, Fall 2026**  
*Last modified: 2026-08-30*

Total: 12 points.

<a id="toc"></a>
## Table of contents

- [Setup](#setup)
- [Problem 1: Coin tossing (2 points)](#problem1)
- [Problem 2: The lighthouse problem (2 points)](#problem2)
- [Problem 3: Bayesian linear regression (4 points)](#problem3)
- [Problem 4: The Bayesian research workflow (4 points)](#problem4)

<a id="setup"></a>
## Setup

Create the directory for data files and import the modules used throughout this
notebook. Run this cell first, and re-run it after any kernel restart.

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>

In [ ]:
import os

import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

import warnings
with warnings.catch_warnings():
    # corner imports arviz, which emits a one-per-day FutureWarning about an
    # upcoming refactor. Nothing here is affected by it.
    warnings.simplefilter("ignore", FutureWarning)
    import emcee
    import corner
    
# Data files are stored in
DATA_DIR = "DataFiles/"

if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

# Figures are stored in
FIGURES_DIR = "Figures/"

if not os.path.exists(FIGURES_DIR):
    os.makedirs(FIGURES_DIR)

rng = np.random.default_rng(seed=2026)

<a id="problem1"></a>
## Problem 1: Coin tossing (2 points)

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>

**Task (see also Yata):** Define a function `bayesian_analysis_coin_flips` (see start code below) that returns the mean, median, and 68%/95% credible intervals of the Bayesian posterior with an input data array of coin flips and using a uniform $[0,1]$ prior for `pH` (probability of heads).

**Task (mainly for testing your code; not on Yata)** Read the data with simulated coin tosses from the file `PS1_Prob1_cointosses.txt` in the `DataFiles` directory.
Each row corresponds to a single toss: 0=tails; 1=heads

Extract the mean and the 68% and 95% credible intervals (Degree-of-belief or DoB intervals) from the first 8 tosses, the first 64 tosses, the first 512 tosses and all 4096 tosses in the data assuming a uniform prior for the probability $p_H$ of obtaining heads in a single toss.

*Hint*: Sample code for computing the DoB interval is available in the lecture notes: "Demonstration: Bayesian Coin Tossing". The tests below will fail if you use a too sparse grid when assigning point esitmates and credible intervals. Use at least 1000 mesh points for evaluating $p(p_H|D,I)$.

**Task (see also Yata):** Check how the width ($d_{95}$) of your 95% credible interval depends on the number of coin tosses ($N$) that is included in the data likelihood. It should be some power-law dependence: $d_{95} \propto N^p$; but which power $p$? What do you expect and what do you find?

*Hint*: You might want to make a logarithmic plot.

In [ ]:
# importing modules

import numpy as np
import matplotlib.pyplot as plt
#...

##################
# YOUR CODE HERE #
##################


In [ ]:
# Read data
data = np.loadtxt(f'{DATA_DIR}/PS1_Prob1_cointosses.txt')

In [ ]:
# Optional. 
# Insert utility code / functions here.

##################
# YOUR CODE HERE #
##################


In [ ]:
# Define a function that returns the mean, median, and 68%/95% credible intervals 
# of the Bayesian posterior with an input data array of coin flips 
# and using a uniform [0,1] prior for the pH (probability of heads).

def bayesian_analysis_coin_flips(data_coin_tosses, verbose=True):
    """
    Returns various Bayesian analysis results for the given data of coin tosses.
    
    The posterior is p( pH | data, I).
    Assume a uniform p(pH|I) = U[0,1] prior
    
    Args:
        data_coin_tosses: Array of shape (m,) with 'm' independent binary data.
            0 = tails; 1 = heads
        verbose: Print the results (default=True)
            
    Returns:
        (mean, mode, median, dob68, dob95): A tuple with the following elements
            mean: The mean of the posterior distribution (float)
            mode: The mode of the posterior distribution (float)
            median: The median of the posterior distribution (float)
            dob68: A tuple (lo,hi) with the lower and upper limits of the 
                68% degree-of-belief range of the posterior distribution (float,float)
            dob95: A tuple (lo,hi) with the lower and upper limits of the 
                95% degree-of-belief range of the posterior distribution (float,float)
    """
##################
# YOUR CODE HERE #
##################


In [ ]:
(mean, mode, median, dob68, dob95) = bayesian_analysis_coin_flips(data[:1])
for output in (mean, mode, median, dob68[0], dob95[0]):
    assert output.dtype=='float64', 'Wrong type'
assert len(dob68)==2, 'DoB tuple should be of length 2'
assert len(dob95)==2, 'DoB tuple should be of length 2'
assert np.abs(mean-0.6667)<0.001
assert np.abs(mode-1.0)<0.001
assert np.abs(median-0.7071)<0.001

# 8 tosses
(mean, mode, median, dob68, dob95) = bayesian_analysis_coin_flips(data[:8])
assert np.abs(mode-0.6250)<0.001
assert np.abs(dob68[0]-0.4462)<0.001
assert np.abs(dob68[1]-0.7531)<0.001
# 1024 tosses
(mean, mode, median, dob68, dob95) = bayesian_analysis_coin_flips(data[:1024])
assert np.abs(median-0.4395)<0.001
# 4096 tosses
(mean, mode, median, dob68, dob95) = bayesian_analysis_coin_flips(data[:4096])
assert np.abs(dob95[0]-0.4625)<0.001
assert np.abs(dob95[1]-0.4931)<0.001

In [ ]:
# Explore the dependence of the width of the 95% DoB on the number of coin tosses.
##################
# YOUR CODE HERE #
##################


<a id="problem2"></a>
## Problem 2: The lighthouse problem (2 points)

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>

A lighthouse is located at position $\alpha$ along a straight shoreline and a distance $\beta$ out at sea. It cannot be seen in the dark, but it emits lightpulses at random intervals (and therefore in random directions). Photodetectors on the shore detect the positions at which the flashes arrive, but they cannot measure the incoming direction. Given the positions of $N$ such flashes
$$
\mathcal{D} = \{ x_k \}_{k=1}^N,
$$
what would you infer for the position $(\alpha, \beta)$ of the lighthouse?

- You can assume that the directions (azimuths) of the flashes are independent and identically distributed random variables given by a uniform distribution $\theta_k \sim \mathcal{U}\left( -\pi/2, \pi/2\right)$.
- Trigonometry relates $\beta \tan\theta_k = (x_k - \alpha)$ and you can use a change of variables to write the likelihood of one observation
  $$
  p(x_k \vert \alpha, \beta, I) = p(\theta_k \vert \alpha, \beta, I) \left| \frac{d\theta_k}{d x_k} \right| = \ldots = \frac{\beta}{\pi\left[ \beta^2 + (x_k - \alpha)^2\right]}.
  $$
- You can use simple, uniform priors for $(\alpha, \beta)$. Assume that the prior ranges are restricted to $-5.0 \leq \alpha \leq +5.0$ and $0.1 \leq \beta \leq 5.0$. 

### (a) Evaluate the (log) posterior. (1 point)
**Tasks (see also Yata)** 
* Define functions `logPrior(theta)`, `logLikelihood(theta, x)` and `logPosterior(theta, x)` that evaluate the corresponding (log) PDFs.

*Note on shapes*: Each function must evaluate many parameter positions in one call. The argument `theta` is an array of shape `(Nsamples, 2)` holding one $(\alpha,\beta)$ pair per row, and the return value is an array of shape `(Nsamples,)` with one number per row. Note that this is a list of positions, not a two-dimensional grid: in task (b) you will build a grid with meshgrid and then reshape it into this form. The data `x` is a one-dimensional array of `Ndata` flash positions, and the likelihood sums the contribution of all of them for each position in `theta`.

Writing these for a single $(\alpha, \beta)$ pair and looping in Python would be far too slow for task (b), where the posterior is evaluated at tens of thousands of positions.

In [ ]:
# Importing modules if needed...

##################
# YOUR CODE HERE #
##################


In [ ]:
# Read data
xpositions = np.loadtxt(f'{DATA_DIR}/PS1_Prob2_xpositions.txt')

In [ ]:
def logPrior(theta):
    '''
    Returns the log uniform prior (-5. <= alpha <= 5., 0.1 <= beta <= 5.) for a grid of theta.
    
    Args
    ----
    theta : ndarray, shape=(Nsamples,2)
        theta[:,0] = alpha, theta[:,1] = beta
        
    Returns
    -------
    logPi : ndarray(Nsamples,)
        log prior (not necessarily normalized) for the grid of theta.
    '''
##################
# YOUR CODE HERE #
##################


def logLikelihood(theta, x):
    '''
    Returns the log likelihood for a grid of theta.
    
    Args
    ----
    theta : ndarray, shape(Nsamples, 2)
        theta[:,0] = alpha, theta[:,1] = beta
    x : nddata, shape=(Ndata,)
        array of positions for measured flashes
        
    Returns
    -------
    logL : ndarray, shape=(Nsanples,) 
        log likelihood (not necessarily normalized) for the grid of theta.
    '''
##################
# YOUR CODE HERE #
##################

def logPosterior(theta, x):
    '''
    Return the log posterior (not necessarily normalized).
    '''
##################
# YOUR CODE HERE #
##################


In [ ]:
# Tests of the prior
assert isinstance(logPrior(np.ones((1,2)))[0], (np.floating, float)), \
    'The output elements of the prior should be floats'
assert logPrior(np.zeros((1,2)))==-np.inf, \
    'The prior = 0 for beta=0.'
assert logPrior(np.ones((5,2))).shape == (5,), \
    'The prior output should be of shape (5,) with 5 input positions.'
# Tests of the likelihood
assert isinstance(logLikelihood(np.ones((1,2)),xpositions[:256])[0], (np.floating, float)), \
    'The output elements of the likelihood should be floats'
assert logLikelihood(np.ones((5,2)),xpositions).shape == (5,), \
    'The likelihood output should be of shape (5,) with 5 input positions and data.'
# Tests of the posterior
assert(np.isclose((logPosterior(np.ones((1,2)),xpositions[:4]) - logPosterior(0.5*np.ones((1,2)),xpositions[:4]) \
                       + logPosterior(np.ones((1,2)),xpositions[:2]) - logPosterior(0.5*np.ones((1,2)),xpositions[:2]))[0], 3.46803166)), \
       'The log posterior has an incorrect dependence on the observations.'
assert(np.isclose((logPosterior(np.ones((1,2)),xpositions) - logPosterior(2*np.ones((1,2)),xpositions))[0], -336.3277888)), \
        'The log posterior has an incorrect dependence on the position.'
# The checks above use theta inside the prior range, where the log prior is a
# constant that cancels in the differences. Probe outside it as well, or a
# posterior that simply forgets the prior would pass.
assert np.all(np.isneginf(logPosterior(np.array([[6.0, 1.0], [0.0, 0.05]]), xpositions))), \
    'The log posterior should be -inf outside the prior range. Did you include the prior?'

### (b) Evaluate the (log) posterior on a grid and find the mode. (1 point)
**Tasks (see also Yata)** 
* Create a grid of $(\alpha, \beta)$ values and evaluate the log posterior for different amounts of available data. The data is loaded into the array `xpositions` in the cell below. Make a 3x3 figure with `subplots` and plot the posterior (not the log!) for the $N = (2, 3, 4, 8, 16, 32, 64, 128, 1024)$ first observations.
* Choose the number of grid points with care. If you evaluate the likelihood by broadcasting, it forms an array of shape (number of grid points x number of observations), so a $D \times D$ grid and $N$ observations needs $N D^2$ floats, and a `numpy` float is 8 bytes. Note that the intermediate steps hold two or three arrays of this size at the same time, so the memory actually used is a few times larger. Size your grid for the largest data set you analyse ($N=1024$), and aim to keep the total below 1 GB. 
* You should use the supplied method `contour_levels` to extract the isoprobability levels that define (0.68, 0.95, 0.997) credible regions. This method takes a 2D grid of PDF values (not necessarily normalized) as an input. These levels can be plotted using the `contour` method.
* Use `scipy.optimize` to find the mode of the posterior for the full set of observations with four significant digits. 

In [ ]:
# Create a grid and evaluate the log-posterior. Plot the bivariate posterior PDF.

# We'll start by defining a function which takes a two-dimensional grid of probability densities and 
# returns 1, 2, and 3-sigma contours. This acts by sorting and normalizing the values and then 
# finding the locations of the  0.682 ,  0.952 , and  0.9972  cutoffs:

def contour_levels(grid):
    """Compute 1, 2, 3-sigma contour levels for a gridded 2D pdf"""
    _sorted = np.sort(grid.ravel())[::-1]
    pct = np.cumsum(_sorted) / np.sum(_sorted)
    cutoffs = np.searchsorted(pct, np.array([0.68, 0.95, 0.997]) ** 2)
    return np.sort(_sorted[cutoffs])

##################
# YOUR CODE HERE #
##################


In [ ]:
# Find the MAP point (maximum a posteriori) for full data set.
#
# You might want to use functionality from scipy to find the mode.
#
##################
# YOUR CODE HERE #
##################


<a id="problem3"></a>
## Problem 3: Bayesian linear regression (4 points)

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>

We will be fitting a linear model (in this case a first order polynomial) to a set data. Our model has two parameters $\vec\theta=[\theta_0,\theta_1]$ (pay attention to the indexing)

$$
y_M(x) = \theta_0 + \theta_1 x 
$$

And our statistical model assumes that errors are independent and identically distributed

$$
y_i = y_M(x_i;\theta) + \varepsilon_i.
$$

Specifically, $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$ and we assume a fixed standard deviation $\sigma = 40$. 

(Note that the $\varepsilon_i \sim \ldots$ notation means that $\varepsilon_i$ is a draw of a random variable that follows the specified distribution.)

The data is generated from a straight line with intercept = 15. and slope = 1.5 plus random noise.

In [ ]:
intercept = 15.
slope = 1.5
theta_true = np.array([intercept, slope])
sigma=40.

Read the data from the file `PS1_Prob3_data.txt`.

In [ ]:
# Load the data and plot with fixed error bar 
# Use np.loadtxt() for loading data (the argument 'unpack=True' is useful)
# and plt.errorbar() for plotting data with errorbars

##################
# YOUR CODE HERE #
##################


### (a) Bayesian linear regression: Implement log prior(s) and likelihood (1 point)

**Task (see also Yata)** Create functions that return the natural logarithm of the likelihood and two different choices of prior. Since we are seeking a parameter posterior you do not have to use proper normalization of the probability densities. 

Consider two different choices for the prior:
1. A uniform prior:
   $$
   p(\theta_0, \theta_1 | I) \propto \left\{ \begin{array}{ll} 1 & \text{if } -100 \le \theta_0 \le 100 \text{ and } -100 \le \theta_1 \le 100 \\ 0 & \text{else} \end{array}\right.
   $$
2. A uniform prior for the intercept and a symmetry-invariant one for the slope
   $$
   p(\theta_0, \theta_1 | I) \propto \left\{ \begin{array}{ll} \frac{1}{(1+\theta_1^2)^{3/2}} & \text{if } -100 \le \theta_0 \le 100  \\ 0 & \text{else}\end{array}\right.
   $$

The second choice is not an arbitrary alternative. It is the form left
invariant when the straight-line model is rewritten with the roles of the axes
exchanged, $x = \theta_1' y + \theta_0'$, so it builds in the fact that neither
variable is privileged, whereas a prior that is uniform in the slope silently
prefers steep lines. The argument is developed in
[Indifferences and translation groups](https://nucleartalent.github.io/LFD_for_Physicists/content/Advanced-Bayesian-methods/assigning-probabilities/sec-assigning-probabilities-i-indifferences-and-translation-groups.html)
in the course literature.


In [ ]:
def log_flat_prior(theta):
    '''
    Returns the log uniform prior (-100 <= theta_i <= 100)
    
    Args:
        theta: array of floats with two elements. theta[0]=intercept. theta[1]=slope
        
    Returns:
        logPi: (float) log prior (not necessarily normalized)
    '''
##################
# YOUR CODE HERE #
##################
    
def log_symmetric_prior(theta):
    r'''
    Returns the log uniform (for theta_0) and symmetric (for theta_1) prior 
    
    (-100 <= theta_0 <= 100; p(theta_1) \propto (1+theta_1^2)^(-3/2))
    
    Args:
        theta: array of floats with two elements. theta[0]=intercept. theta[1]=slope
        
    Returns:
        logPi: (float) log prior (not necessarily normalized)
    '''
##################
# YOUR CODE HERE #
##################


In [ ]:
assert isinstance(log_flat_prior([0.,0.]), (np.floating, float)), 'The output should be a float'
assert isinstance(log_symmetric_prior([0.,0.]), (np.floating, float)), 'The output should be a float'
assert log_flat_prior([0.,0.]) - log_flat_prior([10.,-10.]) == 0., 'The flat prior should be constant in the interval'
assert np.abs(log_symmetric_prior([-10.,1.]) - log_symmetric_prior([20.,2.]) - 1.3744360978112324) <0.001, \
'The log symmetric prior does not evaluate correctly.'

In [ ]:
def log_likelihood(theta, x, y, dy=sigma):
    '''
    Returns the log likelihood.
    
    Args:
        theta: array of floats with two elements. theta[0]=intercept. theta[1]=slope
        x: data (independent variable). array of floats
        y: data (dependent variable). array of floats
        dy: fixed error (optional; default=sigma defined above), standard deviation of a normal distribution
        
    Returns:
        logL: (float) log likelihood
    '''
##################
# YOUR CODE HERE #
##################


In [ ]:
x_test,y_test = np.loadtxt(f'{DATA_DIR}/PS1_Prob3_data.txt',unpack=True)
assert isinstance(log_likelihood([0,0], x_test, y_test), (np.floating, float)), 'The output should be a float'
assert np.abs(log_likelihood([0.,0.], x_test, y_test) - log_likelihood([10.,1.], x_test, y_test) + 6.607647377812501) <0.001, \
'The log likelihood does not evaluate correctly.'

### (b) Use MCMC sampling and plot the posterior for the two different prior choices (1 point)

**Tasks (see also Yata)**
* Use the `emcee` package to sample the posterior with these two different
  priors. Use 32 walkers and 4000 steps each, discarding the first 500 steps
  as warm-up (this gives 112,000 samples per prior).
* Initialize the sampler at starting positions corresponding to samples from the respective prior (see provided code).
* Plot the joint posterior pdf for the two different prior choices using `corner`. Use the keyword argument `truths` to indicate the "true" values of the parameters that generate the data (in the array `theta_true` defined at the start of this problem).
* Which prior shifts the posterior mode, compared to the "true" values, and which way?
* Are the parameters correlated / anti-correlated?

#### A short guide to `emcee`

This is the first time you use `emcee` yourself. While sampling algorithms are
covered in the course, you might have to treat it as a black box for now. A minimal
run takes three steps:

```python
# 1. Initiate the sampler
sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior_function)
# 2. Run the sampler
sampler.run_mcmc(starting_guesses, nsteps, progress=True)
# 3. Collect and flatten the samples
samples = sampler.get_chain(discard=nwarmup, flat=True)
```

* `ndim` is the number of parameters and `nwalkers` the number of chains that
  are advanced in parallel. `emcee` proposes a new position for a walker using
  the positions of the other walkers, so `nwalkers` must be comfortably larger
  than `ndim`.
* `starting_guesses` has shape `(nwalkers, ndim)`: one starting position per
  walker. Use the provided `sample_prior` function below.
* The walkers need some steps to travel from their starting positions to the
  region where the posterior has most of its probability mass. Those first
  samples are not draws from the posterior and are discarded as **warm-up**
  (also called burn-in). That is what `discard=nwarmup` does.
* `flat=True` collapses the chain into a single list of samples of shape
  `(nwalkers*(nsteps-nwarmup), ndim)`. Without it, `get_chain` returns an
  array of shape `(nsteps-nwarmup, nwalkers, ndim)` -- note that the step
  axis comes first.

The settings suggested in the task above are reasonable for this problem; you
are welcome to experiment.

For more detail, see the `emcee` documentation:
[Quickstart](https://emcee.readthedocs.io/en/stable/tutorials/quickstart/) and
[the `EnsembleSampler` class](https://emcee.readthedocs.io/en/stable/user/sampler/).
A worked example on this same straight-line problem is given in
[Fitting a straight line II](https://nucleartalent.github.io/LFD_for_Physicists/content/Bayesian-methods-for-scientific-modeling/exercises-bayesian-methods-for-scientific-modeling/problem-fitting-a-straight-line-ii.html)
in the course literature. 

In [ ]:
def sample_prior(nsamples, kind='symmetric', rng=None):
    """Draw parameter vectors from the prior.

    Args:
        nsamples: int, number of draws
        kind: 'flat' or 'symmetric', the choice of prior for the slope
        rng: numpy Generator

    Returns:
        ndarray of shape (nsamples, ndim)
    """
    rng = np.random.default_rng() if rng is None else rng

    theta0 = rng.uniform(-100., 100., nsamples)          # uniform intercept

    if kind == 'flat':
        theta1 = rng.uniform(-100., 100., nsamples)
    elif kind == 'symmetric':
        # p(theta1) = (1/2) (1 + theta1**2)**(-3/2).  Substituting
        # s = theta1 / sqrt(1 + theta1**2) gives ds/dtheta1 = (1+theta1**2)**(-3/2),
        # so s is uniform on (-1, 1) and we can sample the prior exactly by
        # drawing s and inverting.
        s = rng.uniform(-1., 1., nsamples)
        theta1 = s/np.sqrt(1. - s**2)
    else:
        raise ValueError(f"unknown prior: {kind}")

    columns = [theta0, theta1]
    return np.column_stack(columns)

In [ ]:
# Collect samples from the posterior with the flat prior
##################
# YOUR CODE HERE #
##################


In [ ]:
# Make a corner plot of the posterior with the flat prior.
##################
# YOUR CODE HERE #
##################


In [ ]:
# Collect samples from the posterior with the symmetric prior
##################
# YOUR CODE HERE #
##################


In [ ]:
# Make a corner plot of the posterior with the symmetric prior.
##################
# YOUR CODE HERE #
##################


### (c) Unknown experimental error (1 point)

Repeat the analysis but this time without knowledge of $\sigma$, the standard deviation of the experimental errors. That means that you need to include this width as an unknown (hyper-)parameter, $\sigma_e$, in your statistical model.

- Assume a uniform prior for the width in the range [1, 100]: $\sigma_e \sim \mathcal{U}(1,100)$.
- Restrict the analysis to using only the symmetric prior for the slope.

**Tasks (see also Yata)**
* Use the `emcee` package to sample the posterior with the symmetric prior and
  the unknown error. There are now three parameters. Use 50 walkers and 10,000
  steps each, discarding the first 1000 as warm-up (this gives 450,000
  samples). This sampling budget is provisional; we return to the choice in (d).
* Use the same method as before to sample the prior for the intercept ($\theta_0$) and the slope ($\theta_1$). You need to stack a column with samples from the prior for the experimental error  ($\sigma_e$) to get a complete specification of starting positions.
* Plot the joint posterior pdf using `corner`. Use the keyword argument `truths` to indicate the "true" values of the parameters that generate the data (the model parameters are in the array `theta_true` and the noise that was used is in the variable `sigma`).
* Compare the marginal posterior distributions for the slope and the intercept
  with those from (b), where the width was fixed at its true value. Do the
  credible intervals become narrower or wider once $\sigma_e$ is treated as
  unknown, and why?

In [ ]:
# Collect samples from the posterior with the symmetric prior and the unknown width
##################
# YOUR CODE HERE #
##################


In [ ]:
# Make a corner plot of the posterior with the symmetric prior and the unknown width
##################
# YOUR CODE HERE #
##################


### (d) How many samples is a chain actually worth? (1 point)

In task (c) you were asked for 450,000 samples. This subtask explains why.

A chain of $N$ MCMC samples does *not* provide $N$ independent draws, because
consecutive samples are correlated. The **effective sample size**
$N_\mathrm{eff} = N/\tau$, where $\tau$ is the **integrated autocorrelation
time**, is roughly the number of independent draws the chain is worth. Anything
you estimate from the chain---a posterior mean, a credible-interval endpoint---carries a Monte Carlo uncertainty set by $N_\mathrm{eff}$, not by $N$. For example, you should be able to estimate the Monte Carlo uncertainty of a posterior mean from the standard deviation of the samples divided by the square root of $N_\mathrm{eff}$:

$$
\mathrm{Std}\big[\hat{\mathbb{E}}[\theta_0]\big] \approx
\frac{\mathrm{Std}[\theta_0]}{\sqrt{N_\mathrm{eff}}} .
$$

You will compare two *proposal strategies* on this basis. You are not expected
to know how either one works; sampling algorithms are covered later in the
course. In `emcee` they are selected with the `moves` keyword, and nothing else
about the run changes:

```python
rw      = emcee.moves.GaussianMove(20.0**2)   # random-walk Metropolis
stretch = emcee.moves.StretchMove()           # emcee's default
```

Note that `emcee` ignores any `Generator` you create with
`np.random.default_rng`. When the sampler is constructed it copies NumPy's
*legacy* global random state, the one that `np.random.seed` sets. A run is
therefore reproducible only if you seed that legacy state or set
`sampler.random_state` explicitly. The start code below does the latter for
you, as do the samplers earlier in this problem.

**Tasks (see also Yata)**

1. Sample the posterior of task (c) with each of the two moves, using 32
   walkers and 5000 steps, discarding the first 500 steps as burn-in. Repeat
   each run for **16 different random seeds**. Pass your own log-posterior
   function from task (c) to `sample_with_move`, together with any extra
   arguments it takes besides `theta`.
2. Fix the **starting positions** such that the observed scatter is only due to sampling noise.
3. From every run record the posterior mean of the intercept $\theta_0$. For each move, report
   **the mean and the standard deviation across the 16 seeds**. Every run used
   the same data, the same posterior and the same budget, so this scatter is
   pure Monte Carlo noise.
4. Obtain $\tau$ for $\theta_0$ from `sampler.get_autocorr_time(discard=nburn)` and form
   $N_\mathrm{eff}$. Check whether $\mathrm{Std}[\theta_0]/\sqrt{N_\mathrm{eff}}$
   reproduces the scatter you measured in step 3.
5. For one of the two moves, `get_autocorr_time()` raises an `AutocorrError`
   instead of returning a number. You should pass the argument `quiet=True` to downgrade the error to a log message and get the estimate anyway. But read the message and think whether you expect the estimate to be too large or
   too small.

In [ ]:
def sample_with_move(move, seed, log_posterior, args=(),
                     nwalkers=32, nsteps=5000, nburn=500):
    '''Sample the task (c) posterior with a given emcee move.

    Args:
        move: an emcee move object, e.g. emcee.moves.StretchMove()
        seed: int, controls the sampling only. The walker start positions are
            drawn from a separate, fixed generator (`rng_start` below) so that
            the scatter across seeds is due to sampling noise alone.
        log_posterior: your log-posterior function from task (c)
        args: tuple of extra arguments passed to `log_posterior` after theta,
            e.g. `(x, y)`. Leave empty if it takes theta alone.
        nwalkers, nsteps, nburn: int

    Returns:
        (flat_chain, tau): the flattened chain of shape (n, 3) with the burn-in
        discarded, and the integrated autocorrelation time of theta_0
        (estimated with quiet=True so that a too-short chain does not raise).
    '''
    ndim = 3

    # Use this RNG for the start positions (fixed, does not depend on `seed`)
    rng_start = np.random.default_rng(0)
    
    sampler_d = emcee.EnsembleSampler(nwalkers, ndim, log_posterior,
                                      args=args, moves=move)
    # emcee copies numpy's legacy global random state at construction; pin it
    # down so that the comparison is reproducible.
    sampler_d.random_state = np.random.RandomState(seed).get_state()

##################
# YOUR CODE HERE #
##################


In [ ]:
# Compare the two moves over 16 seeds.
##################
# YOUR CODE HERE #
##################


**Questions**

1. Both chains have the same length. In what sense is one of them "longer"?
2. Does the $\sqrt{N_\mathrm{eff}}$ prediction work equally well for both
   moves? If not, connect the discrepancy to your answer in task 5 above.
3. You double the chain length, keeping everything else fixed. What can you expect for a Monte Carlo estimate of a posterior mean?

<a id="problem4"></a>
## Problem 4: The Bayesian research workflow (4 points)

In this problem you will carry out a complete Bayesian analysis of a small
physics experiment, following the four-step workflow of the lecture notes:

1. Formulate informative priors *before* the new data is used.
2. Define a statistical model relating the physics model and the data,
   including all errors.
3. Compute the posterior probabilities.
4. Do model checking.

The emphasis is on steps 1 and 4, which we have not yet practised. Both rely
on the same tool: a **predictive distribution**, i.e. a distribution over
*predicted data* rather than over parameters. Applied before seeing the measured data it is the
*prior predictive distribution* and it tells you whether your priors describe
a believable experiment. Applied after the inference it is the *posterior
predictive distribution* and it tells you whether your model can reproduce
the data you actually got.

We will use `emcee` as a black box to collect samples from the posterior. You are not
expected to know how it works -- sampling algorithms are covered later in
the course.

<div style="text-align: right"><a href="#toc">&#8593; Table of contents</a></div>

### The experimental setup and the statistical model

A radioactive sample is placed in front of a detector at $t=0$ and the number
of recorded decays is counted in consecutive time bins of width
$\Delta t = 2\,$s, out to $t = 120\,$s. The detector also registers a constant
background from cosmic rays and from activity in the surrounding material.

Our physics model for the expected count *rate* is a single decaying species
on top of that background,

$$
r(t; \boldsymbol{\theta}) = S\,e^{-t/\tau} + b ,
\qquad \boldsymbol{\theta} = (S, \tau, b),
$$

where $S$ is the initial count rate of the sample [s$^{-1}$], $\tau$ its mean
lifetime [s], and $b$ the background rate [s$^{-1}$]. The expected number of
counts in the bin centred at $t_i$ is then
$\mu_i(\boldsymbol{\theta}) = r(t_i;\boldsymbol{\theta})\,\Delta t$.

Counting decays is a Poisson process, so our statistical model for $y_i$, the
number of counts actually recorded in the bin centred at $t_i$, is

$$
y_i \sim \mathrm{Poisson}\!\left(\mu_i(\boldsymbol{\theta})\right),
$$

with the counts in different bins independent. The data is thus a list of 60
integers, one per bin. Note that there is no free "error bar" here: the
noise model follows from the physics of counting.

In [ ]:
# The measurement schedule is part of the experimental design and is therefore
# known before any data is taken: 2 s bins covering the first 120 s.
dt = 2.0
t_edges = np.arange(0., 120. + dt, dt)
t_bins = 0.5*(t_edges[:-1] + t_edges[1:])     # bin centres

print(f'{len(t_bins)} bins of width {dt} s, covering 0 to {t_edges[-1]:.0f} s')

### (a) Prior elicitation and prior predictive checking (1 point)

Before looking at the data we must say what we knew beforehand. Here is the
available background information $I$:

* **Lifetime.** Nuclear systematics and an older measurement on a neighbouring
  isotope suggest a mean lifetime of a few tens of seconds. The old value was
  $\tau \approx 30\,$s, but it is not to be trusted to better than a factor
  of about two.
* **Source strength.** The supplier quotes an initial activity "of order
  $10^2$ decays per second", uncertain by a factor of a few. The detector
  efficiency and solid angle are only roughly known.
* **Background.** A dedicated calibration run with no sample present recorded
  $N_\mathrm{bkg} = 305$ counts in $T_\mathrm{bkg} = 100\,$s.
* **Detector.** Because of its dead time the detector cannot record more
  than about $10^4$ counts per second. Above that it saturates and the
  recorded counts are meaningless.

A quantity known only up to a multiplicative factor is naturally described by
a **log-normal** prior, which is why we use it for $S$ and $\tau$ below. The
calibration run gives a well-determined background: with $N_\mathrm{bkg}$
counts in $T_\mathrm{bkg}$ seconds, $\hat b = N_\mathrm{bkg}/T_\mathrm{bkg}$
with standard error $\sqrt{N_\mathrm{bkg}}/T_\mathrm{bkg}$, which we encode as
a normal prior.

**Task (see also Yata):** Implement the two candidate priors below.

1. A **weakly informative** prior that encodes the information above:
   $$
   S \sim \mathrm{LogNormal}(\ln 150,\ (\ln 3)^2), \quad
   \tau \sim \mathrm{LogNormal}(\ln 30,\ (\ln 2)^2), \quad
   b \sim \mathcal{N}(\hat b,\ \hat\sigma_b^2).
   $$
2. A **diffuse** prior, of the kind that is often chosen out of habit in order
   to "let the data speak". Here we choose a prior that is diffuse in $S$ and $\tau$:
   $$
   S \sim \mathcal{U}(0, 10^6), \quad
   \tau \sim \mathcal{U}(0, 10^4), \quad
   b \sim \mathcal{N}(\hat b,\ \hat\sigma_b^2).
   $$

We keep the same background prior in both cases so that the comparison isolates
the effect of the $S$ and $\tau$ priors. Both functions should return
$-\infty$ outside the support; all three parameters are rates or a time and
must therefore be positive. Use `scipy.stats.lognorm(s=sigma, scale=np.exp(mu))`.

In [ ]:
# The background calibration run is prior information: it was taken before
# the sample was measured, and it is what makes the prior on b informative.
N_bkg = 305
T_bkg = 100.0

b_hat = N_bkg / T_bkg
b_sig = np.sqrt(N_bkg) / T_bkg


def log_prior_weakly_informative(theta):
    '''Log of the weakly informative prior for theta = (S, tau, b).'''
    S, tau, b = theta
##################
# YOUR CODE HERE #
##################


def log_prior_diffuse(theta):
    '''Log of the (S,b)-diffuse prior for theta = (S, tau, b).'''
    S, tau, b = theta
##################
# YOUR CODE HERE #
##################


In [ ]:
# Checkpoints
assert np.isfinite(log_prior_weakly_informative([150., 30., 3.]))
assert log_prior_weakly_informative([-1., 30., 3.]) == -np.inf
assert log_prior_diffuse([1e7, 30., 3.]) == -np.inf
assert log_prior_weakly_informative([150., 30., 3.]) > \
       log_prior_weakly_informative([150., 300., 3.]), \
       'tau = 300 s should be less probable a priori than tau = 30 s'

**Task (see also Yata):** Now check what these priors imply about the *data*.

Note that we have not yet looked at the measurement. That is the point: a prior
predictive check asks whether your priors describe experiments that could
plausibly happen *before* you have anything to compare them with. The only
things you may use are the model, the measurement schedule, and the background
information $I$ above.

Write a function `prior_predictive_sample` that draws `n_draws` parameter
vectors from a given prior and, for each one, simulates a complete replicated
dataset by drawing Poisson counts. It should return an array of shape
`(n_draws, len(t_bins))`.

Then, for each of the two priors:

* Plot a sample of the replicated count curves. Use a logarithmic count axis.
* Plot the distribution of the total number of counts $\sum_i y_i$, and mark
  the range $10^3$ to $10^4$ that the background information leads us to
  expect: an initial rate of order $10^2\,$s$^{-1}$ decaying over a few tens of
  seconds, plus a background near $3\,$s$^{-1}$ for the remainder of the
  $120\,$s run.
* Compute, for each prior, the fraction of draws that describe an experiment
  which is **physically possible** (peak rate below the $10^4\,$s$^{-1}$
  detector limit) and the fraction that is **informative about $\tau$** (count
  rate falls by at least a factor of ten between the first and the last bin).
* For the plots, a few hundred prior samples are sufficient. For the fractions, you could use a few thousand samples.

In [ ]:
# Helper function
def draw_from_prior(kind, n_draws, rng):
    '''Draw parameter vectors (S, tau, b) from the named prior.'''
##################
# YOUR CODE HERE #
##################

# Helper function
def expected_counts(theta, t):
    '''Expected counts mu_i in each bin, for theta = (S, tau, b).'''
##################
# YOUR CODE HERE #
##################


def prior_predictive_sample(kind, n_draws, rng):
    '''Replicated datasets drawn from the prior predictive distribution.'''
##################
# YOUR CODE HERE #
##################


In [ ]:
# Checkpoints
_test = prior_predictive_sample('weakly informative', 7, np.random.default_rng(1))
assert _test.shape == (7, len(t_bins)), 'wrong shape of the replicated datasets'
assert _test.dtype.kind in 'iu', 'counts should be integers'
assert np.isclose(expected_counts([200., 25., 3.], np.array([0.]))[0], 406.), \
       'check the expected number of counts at t = 0'

In [ ]:
##################
# YOUR CODE HERE #
##################


In [ ]:
# How many of the prior draws describe a sensible experiment?
##################
# YOUR CODE HERE #
##################


**Question.** Comment on the two prior predictive distributions, using only
what you knew before the measurement.

* Which of the two priors describes experiments that this apparatus could
  actually perform, and what does the other one predict?
* Look at the shape of the replicated curves under the diffuse prior. Would an
  experiment of that kind tell you anything about $\tau$? Explain the
  connection to the range of $\tau$ that the diffuse prior allows.
* The diffuse prior was not chosen to be absurd -- it was chosen to be
  *uninformative*. Explain in your own words why "uninformative" and
  "harmless" are not the same thing.

### The experiment is performed

The prior predictive check is complete, so we may now look at the measurement.
The file contains one row per time bin: the bin edges, the bin centre, and the
number of counts recorded in that bin.

In [ ]:
data = np.loadtxt(DATA_DIR + 'PS1_Prob4_decay_data.txt')
t_lo, t_hi, t_data, y_data = data.T
y_data = y_data.astype(int)

# the measurement followed the schedule assumed in part (a)
assert np.allclose(t_data, t_bins), 'unexpected time bins in the data file'

print(f'{len(y_data)} bins of width {dt} s, {y_data.sum()} counts in total')

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(t_data, y_data, yerr=np.sqrt(y_data), fmt='o', ms=3, color='k')
ax.set(xlabel=r'$t$ [s]', ylabel=f'counts per {dt:.0f} s bin', yscale='log')
fig.tight_layout()

### (b) The posterior, and when the prior matters (1 point)

**Task (see also Yata):** Implement the log-likelihood for the Poisson
statistical model. Dropping the $y_i!$ term, which does not depend on the
parameters,

$$
\ln \mathcal{L}(\boldsymbol{\theta}) =
\sum_i \left[ y_i \ln \mu_i(\boldsymbol{\theta}) - \mu_i(\boldsymbol{\theta})\right].
$$

Then build the log-posterior and sample it with `emcee` using the weakly
informative prior. Show a corner plot and report the median and 68% credible
interval for $\tau$.

In [ ]:
def log_likelihood(theta, t, y):
    '''Poisson log-likelihood, up to a parameter-independent constant.'''
##################
# YOUR CODE HERE #
##################


def log_posterior(theta, t, y, log_prior=log_prior_weakly_informative):
    '''Log-posterior for theta = (S, tau, b).'''
##################
# YOUR CODE HERE #
##################


In [ ]:
# Checkpoints
# (note: we dropped the ln(y_i!) term, so this log-likelihood is not negative)
assert log_likelihood([-1., 25., 3.], t_data, y_data) == -np.inf
assert np.isclose(log_likelihood([200., 25., 3.], t_data, y_data), 25804.222, rtol=1e-5)
assert log_likelihood([200., 25., 3.], t_data, y_data) > \
       log_likelihood([200., 80., 3.], t_data, y_data), \
       'the data should prefer tau = 25 s over tau = 80 s'

In [ ]:
def run_emcee(log_prior, t, y, nwalkers=32, nsteps=4000, nburn=1000, seed=2026):
    '''Sample the posterior with emcee. Returns a flat chain of shape (n, 3).'''
    rng_local = np.random.default_rng(seed)
    ndim = 3
    start = np.array([150., 30., b_hat]) * \
        (1 + 0.05 * rng_local.standard_normal((nwalkers, ndim)))
    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_posterior,
                                    args=(t, y, log_prior))
    # emcee copies numpy's legacy global state at construction; pin it so the
    # run does not depend on a seed set elsewhere in the notebook.
    sampler.random_state = np.random.RandomState(seed).get_state()
    sampler.run_mcmc(start, nsteps, progress=False)
    return sampler.get_chain(discard=nburn, flat=True)


##################
# YOUR CODE HERE #
##################


**Task (see also Yata):** Now repeat the inference with the *diffuse* prior,
first using the full dataset and then using **only the first five bins**
($t \le 9\,$s), as if the experiment had been stopped early. Report the median
and 68% credible interval for $\tau$ in all four cases and collect them in a
small table.

In [ ]:
##################
# YOUR CODE HERE #
##################


### (c) Posterior predictive checking (2 points)

Step 4 of the workflow. We now have a posterior, but nothing so far has told
us whether the model is any good. A posterior predictive check asks a simple
question: if we simulate new experiments using the parameters we just
inferred, do they look like the experiment we actually ran?

The posterior predictive distribution for a replicated dataset
$\tilde{\boldsymbol{y}}$ is obtained by marginalising over the posterior,

$$
p(\tilde{\boldsymbol{y}} \mid \boldsymbol{y}, I) =
\int p(\tilde{\boldsymbol{y}} \mid \boldsymbol{\theta}, I)\,
      p(\boldsymbol{\theta} \mid \boldsymbol{y}, I)\, d\boldsymbol{\theta},
$$

which in practice means: for each posterior sample $\boldsymbol{\theta}^{(k)}$,
generate one replicated dataset. The list of replicated datasets then constitue samples from the posterior predictive distribution $p(\tilde{\boldsymbol{y}} \mid \boldsymbol{y}, I)$.

**Task (see also Yata):** Generate replicated datasets from the posterior
predictive distribution (using the weakly informative prior chain) and
overlay their 68% and 95% envelopes on the observed data. Return both the
replicated datasets and the parameter vectors that generated them: the second
test statistic below is evaluated at each posterior draw and needs them.

In [ ]:
def posterior_predictive_sample(chain, n_draws, rng):
    '''Replicated datasets, and the parameters that generated them, drawn from
    the posterior predictive distribution.'''
##################
# YOUR CODE HERE #
##################


In [ ]:
##################
# YOUR CODE HERE #
##################


**Task (see also Yata):** A visual check is suggestive but not quantitative.
Instead, use a **test statistic** that is a function of data, $T(\boldsymbol{y})$, and compute the *posterior
predictive p-value*

$$
p_B = \Pr\!\left[\, T(\tilde{\boldsymbol{y}}) \ge T(\boldsymbol{y}) \,\right],
$$

i.e. the fraction of replicated datasets whose statistic is at least as
extreme as the observed one. A value close to 0 or 1 signals that the model
cannot reproduce that feature of the data. Use both of the following test statistics:

1. $T_\mathrm{late}(\boldsymbol{y}) = \sum_{t_i > 90\,\mathrm{s}} y_i$, the
   number of counts in the tail of the measurement.
2. The Pearson statistic
   $T_{\chi^2}(\boldsymbol{y};\boldsymbol{\theta}) =
   \sum_i (y_i - \mu_i)^2 / \mu_i$, which is an omnibus test$^*$. Note that this
   one depends on $\boldsymbol{\theta}$, so it must be evaluated at each
   posterior draw for both the observed and the replicated dataset.

Plot the distribution of $T(\tilde{\boldsymbol{y}})$ with the observed value
marked, for both statistics.

${}^*$*Omnibus test*: In statistics, an omnibus test is one that asks a single undirected question: "is anything wrong?"; rather than testing a specific, named way in which the model might fail.

In [ ]:
##################
# YOUR CODE HERE #
##################


**Discussion questions** (not tested) 

1. Your two statistics do not agree: one of them rejects the model decisively,
   the other does not. Which is which? Explain why a statistic aimed
   specifically at the late-time bins is more sensitive here than an omnibus
   statistic summed over all 60 bins. (This is the central practical lesson of
   posterior predictive checking: a check is only as good as the statistic you
   choose, and a model can fail in a way that an undirected test dilutes to
   invisibility.)
2. Look again at the residual panel in the figure above. Where in time does
   the model fail, and in which direction?
3. Propose a modification of the *physics* model that would explain this
   pattern. Which step of the four-step workflow does your proposal send you
   back to?